In [3]:
%load_ext autoreload
%autoreload 2
import sys
import os
import time
import pickle
import json
import csv
import pandas as pd

# notebooks/ から見た src のパスを追加
sys.path.append(os.path.abspath("../src"))

# 自作パッケージを import
from sim_utils import *
import pymdp

# result = load_object_from_pickle()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
%load_ext autoreload
%autoreload 2
from sim_utils import Optimizer
import inspect


sim_dir = Path("/home/rikut/py_venvs/my_sim_project/stimuli_final/final_sim_dof3_20251217_021445")
results = load_simulation_data(sim_dir)

# --- Config 関連 ---
config_json = results["config"].get("config_json")
robot = results["config"].get("robot")
agent = results["config"].get("agent")
env = results["config"].get("env")
const_beliefs_result = results["config"].get("const_beliefs")
optimizer_state = results["config"].get("optimizer_state")

# --- Optimizer の再構築 ---
if optimizer_state is not None:
    # 1. Optimizer.__init__ が受け取れる引数リストを取得
    sig = inspect.signature(Optimizer.__init__)
    valid_keys = sig.parameters.keys() # ['self', 'robot', 'agent', 'timesteps', ...]

    # 2. optimizer_state から、__init__ に渡せる有効なデータだけを抽出
    # ただし 'robot' と 'agent' は既に別途ロードしているので、そちらを優先する
    init_params = {
        k: v for k, v in optimizer_state.items() 
        if k in valid_keys and k not in ['self', 'robot', 'agent', 'env']
    }

    # 3. 再構築
    # ロード済みの robot, agent と、抽出したパラメータを組み合わせてインスタンス化
    opt = Optimizer(
        robot=robot, 
        agent=agent, 
        **init_params
    )

    # 4. 状態の復元（__init__ で初期化されない、計算済みの結果など）
    if "const_beliefs" in optimizer_state:
        opt.const_beliefs = optimizer_state["const_beliefs"]
    if "const_beliefs_result" in optimizer_state:
        opt.const_beliefs_result = optimizer_state["const_beliefs_result"]
    
    # 以前の最適化結果（もしあれば）も復元
    opt.best_cost = optimizer_state.get("best_cost")
    opt.best_particle = optimizer_state.get("best_particle")

    print(f"Optimizer reconstructed successfully with {init_params.keys()}")
else:
    opt = results["config"].get("used_optimizer")

# 確認：メソッドが存在するかチェック
if hasattr(opt, 'hybrid_optimize'):
    print("hybrid_optimize is now available!")
else:
    print("Error: hybrid_optimize is still missing...")
    

# --- Landscape 関連 ---
df = results["landscape"].get("df")

# raw_data (辞書) の中身を個別の変数に展開
raw_data = results["landscape"].get("raw_data", {})
if raw_data:
    energies = raw_data.get('energy')
    jerks = raw_data.get('jerk')
    torque_changes = raw_data.get('torque_change')
    vfes = raw_data.get('vfe')
    klds = raw_data.get('kld')
    bss = raw_data.get('bs')
    uns = raw_data.get('un')
    qss = raw_data.get('qs')

# --- Stimuli 関連 ---
targets_list_LE = results["stimuli"].get("ig_energy")
targets_list_LVFE = results["stimuli"].get("ig_vfe")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Optimizer reconstructed successfully with dict_keys(['timesteps', 'dt', 'start', 'end', 'limits', 'num_knots', 'compensate_grav'])
hybrid_optimize is now available!


In [8]:

num_knots = opt.num_knots

iter = 491 # 先行研究では491?(varの間隔が0.001)

varmin = 0.01 * (2*np.pi) # 先刻研究通りなら0.01(ただし制御点は0~1の値域であったので、ここでは関節角度にスケールするべきか)
varmax = 0.5 * (2*np.pi) # 先刻研究通りなら0.5(ただし制御点は0~1の値域であったので、ここでは関節角度にスケールするべきか)

n_divisions = iter
var_list = np.linspace(varmin, varmax, n_divisions, endpoint=True)

# 基準とする動き
base_particle = opt.const_beliefs_result.best_particle
particle_hist = []

# optimizer
model = opt.model
time_steps = opt.timesteps
start = opt.start
end = opt.end
data = opt.data
dt = opt.dt
compensate_grav = False # Falseで重力考慮、Trueで無重力
env = opt.env
const_beliefs = opt.const_beliefs
limits = opt.limits

pbar = tqdm(range(iter), desc="シミュレーション中", position=1)
for i in pbar:
    start_time = time.time()

    # ランダムな動きを生成し、環境に設定
    # qs = random_qs_spline(model, time_steps, start, end, seed=i, num_knots=num_knots)
    # opt.env.computeAllobs(qs, dt)

    # もととなる動きにノイズを加える形で生成するバージョン
    var = var_list[i]
    (qs ,grads), particle= random_qs_from_base(model, time_steps, 
                                   start, end, 
                                   base_particle=base_particle, 
                                   var=var, limits=limits, seed=i, 
                                   type="B-spline", 
                                   grad1=True, grad2=True, grad3=True,
                                   particle_return=True)
    particle_hist.append(particle)

all_particles = particle_hist

def pick_best_from_landscape(df, targets_list, tolerance=0.1):
    rows = []
    
    for target_id, targets in targets_list.items():
        target_ig = targets.get('ig', {}).get('target')
        
        # 1. 指定した範囲内のデータをフィルタリング
        mask = (df['ig'] >= target_ig - tolerance) & (df['ig'] <= target_ig + tolerance)
        candidates = df[mask]
        
        if not candidates.empty:
            # 範囲内の中から、Energyが最小のものを選択
            best_match = candidates.sort_values('energy').iloc[0]
            success = True
        else:
            # 範囲内にない場合は、単純に最もIGが近いものを1つ選択
            best_match = df.loc[(df['ig'] - target_ig).abs().idxmin()]
            success = False # ターゲット範囲外なのでFalseとする
            
        # 2. 既存の関数と同じ形式で辞書を作成
        row = {
            'target_id': target_id,
            'original_index': best_match.name,
            # --- 抽出された結果 (Achieved) ---
            'ig': best_match.get('ig'),
            'energy': best_match.get('energy'),
            'vfe': best_match.get('vfe'),
            
            # --- 目標としていた値 (Target) ---
            'target_ig': target_ig,
            'target_energy': targets.get('energy', {}).get('target'),
            'target_vfe': targets.get('vfe', {}).get('target'),
            
            # --- 付加情報 (ピックアップ版として調整) ---
            'success': success and (best_match.get('vfe') < 1.0), # VFEも考慮
            'nit': 0, # 最適化ループ回数ではないので0
            'best_cost': best_match.get('energy'), # 最小化対象はEnergy
            'robot_model': 'from_landscape', # どこから来たか識別用
            'optimizer': 'Landscape_Picker' # 手法の識別用
        }
        rows.append(row)
        
    return pd.DataFrame(rows)


シミュレーション中: 100%|██████████| 491/491 [00:00<00:00, 3316.69it/s]


In [9]:
import pandas as pd
import numpy as np

# --- 1. VFEに基づいた3つのインデックスを特定 ---
vfe_idx_min = df['vfe'].idxmin()
vfe_idx_max = df['vfe'].idxmax()

# 中間値（Median）に最も近いインデックスを探す
vfe_median_value = df['vfe'].median()
vfe_idx_mid = (df['vfe'] - vfe_median_value).abs().idxmin()

# ターゲットの情報をリスト化
vfe_targets = [
    {'id': 'vfe_min', 'idx': vfe_idx_min, 'label': 'Minimum VFE (Most Stable)'},
    {'id': 'vfe_mid', 'idx': vfe_idx_mid, 'label': 'Median VFE (Intermediate)'},
    {'id': 'vfe_max', 'idx': vfe_idx_max, 'label': 'Maximum VFE (Least Stable)'}
]

# --- 2. 保存用設定 ---
vfe_study_dir = sim_dir / "stimuli/practice"
vfe_study_dir.mkdir(parents=True, exist_ok=True)

picked_vfe_rows = []

# --- 3. 抽出と動画生成 ---
for item in vfe_targets:
    target_id = item['id']
    idx = int(item['idx'])
    
    # データの特定
    particle = all_particles[idx]
    qs = qss[idx]
    
    # メトリクスの取得と付加情報の追加
    row = df.loc[idx].copy()
    row['vfe_category'] = target_id
    row['description'] = item['label']
    picked_vfe_rows.append(row)
    
    # 動画の生成
    print(f"Generating video for {target_id} (Index: {idx}, VFE: {row['vfe']:.4f})...")
    plot_robot_motion(
        qs=qs, 
        opt=opt, 
        folder_path=vfe_study_dir, 
        file_name=f"{target_id}.mp4"
    )

# --- 4. 結果の保存 ---
df_vfe_analysis = pd.DataFrame(picked_vfe_rows)
df_vfe_analysis.index = [item['id'] for item in vfe_targets]

df_vfe_analysis.to_pickle(vfe_study_dir / "practice_metrics.pkl")
df_vfe_analysis.to_csv(vfe_study_dir / "practice_metrics.csv")

print(f"\n--- VFE Analysis Selection ---")
print(df_vfe_analysis[['vfe', 'ig', 'energy', 'description']])

2025-12-22 05:01:43,152 - matplotlib.animation - INFO - Animation.save using <class 'matplotlib.animation.FFMpegWriter'>
2025-12-22 05:01:43,154 - matplotlib.animation - INFO - MovieWriter._run: running command: ffmpeg -f rawvideo -vcodec rawvideo -s 640x480 -pix_fmt rgba -framerate 30.0 -loglevel error -i pipe: -vcodec h264 -pix_fmt yuv420p -y /home/rikut/py_venvs/my_sim_project/stimuli_final/final_sim_dof3_20251217_021445/stimuli/practice/vfe_min.mp4


Generating video for vfe_min (Index: 3, VFE: 205.9353)...
Saving animation as '/home/rikut/py_venvs/my_sim_project/stimuli_final/final_sim_dof3_20251217_021445/stimuli/practice/vfe_min.mp4'...


2025-12-22 05:01:46,838 - matplotlib.animation - INFO - Animation.save using <class 'matplotlib.animation.FFMpegWriter'>
2025-12-22 05:01:46,839 - matplotlib.animation - INFO - MovieWriter._run: running command: ffmpeg -f rawvideo -vcodec rawvideo -s 640x480 -pix_fmt rgba -framerate 30.0 -loglevel error -i pipe: -vcodec h264 -pix_fmt yuv420p -y /home/rikut/py_venvs/my_sim_project/stimuli_final/final_sim_dof3_20251217_021445/stimuli/practice/vfe_mid.mp4


Saved successfully.
Generating video for vfe_mid (Index: 347, VFE: 445.3164)...
Saving animation as '/home/rikut/py_venvs/my_sim_project/stimuli_final/final_sim_dof3_20251217_021445/stimuli/practice/vfe_mid.mp4'...


2025-12-22 05:01:50,086 - matplotlib.animation - INFO - Animation.save using <class 'matplotlib.animation.FFMpegWriter'>
2025-12-22 05:01:50,087 - matplotlib.animation - INFO - MovieWriter._run: running command: ffmpeg -f rawvideo -vcodec rawvideo -s 640x480 -pix_fmt rgba -framerate 30.0 -loglevel error -i pipe: -vcodec h264 -pix_fmt yuv420p -y /home/rikut/py_venvs/my_sim_project/stimuli_final/final_sim_dof3_20251217_021445/stimuli/practice/vfe_max.mp4


Saved successfully.
Generating video for vfe_max (Index: 482, VFE: 738.7921)...
Saving animation as '/home/rikut/py_venvs/my_sim_project/stimuli_final/final_sim_dof3_20251217_021445/stimuli/practice/vfe_max.mp4'...
Saved successfully.

--- VFE Analysis Selection ---
                vfe          ig       energy                 description
vfe_min  205.935275   58.119906    37.733851   Minimum VFE (Most Stable)
vfe_mid  445.316445  144.644354   757.758590   Median VFE (Intermediate)
vfe_max  738.792092  142.800558  7516.781083  Maximum VFE (Least Stable)


In [10]:
pathstr = "/home/rikut/py_venvs/my_sim_project/results/sim_dof2_20251215_213121/config/const_beliefs_result.pkl"
res_path = Path(pathstr)
res_folder = res_path.parent
try:
    with open(res_path, 'rb') as f:
        res = pickle.load(f)
    print(f"Loaded:")
except Exception as e:
    print(f"Failed to load : {e}")

plot_robot_motion(
    res) 
    # folder_path=res_folder, 
    # file_name = res_path.with_suffix('.mp4'))

2025-12-18 16:32:14,536 - matplotlib.animation - INFO - Animation.save using <class 'matplotlib.animation.HTMLWriter'>


Loaded:
Now preparing HTML display...


In [11]:
res_path

PosixPath('/home/rikut/py_venvs/my_sim_project/results/sim_dof2_20251215_213121/config/const_beliefs_result.pkl')

In [7]:
# フォルダ内のすべてのpklを読み込んで処理する
folder = "/home/rikut/py_venvs/my_sim_project/storage/final_sim_dof3_20251217_021445/stimuli/ig_energy"
folder_path = Path(folder)

loaded_data = []
for file_path in folder_path.iterdir():
    if file_path.is_file():
            
            # 2. 拡張子がないかチェック
            # .suffix が空文字列（''）であれば拡張子がない
            if file_path.suffix == "":
                
                # 拡張子がないファイルをpklとしてロード
                print(f"ロード対象ファイル: {file_path.name}")
                
                try:
                    with open(file_path, 'rb') as f:
                        result = pickle.load(f)
                        loaded_data.append(result)
                    print("--- ロード完了 ---")

                    file_rename = file_path.with_suffix('.mp4')
                    plot_robot_motion(result, folder_path=folder, file_name=file_rename)
                        
                except pickle.UnpicklingError as e:
                    # pklファイルとして無効な場合のエラー処理
                    print(f"警告: ファイル '{file_path.name}' は有効なPKLファイルではありません。スキップします。エラー: {e}")
                except Exception as e:
                    # その他のファイル読み取りエラー処理
                    print(f"警告: ファイル '{file_path.name}' の読み込み中に予期せぬエラーが発生しました。スキップします。エラー: {e}")



2025-12-17 12:04:10,475 - matplotlib.animation - INFO - Animation.save using <class 'matplotlib.animation.FFMpegWriter'>
2025-12-17 12:04:10,477 - matplotlib.animation - INFO - MovieWriter._run: running command: ffmpeg -f rawvideo -vcodec rawvideo -s 640x480 -pix_fmt rgba -framerate 30 -loglevel error -i pipe: -vcodec h264 -pix_fmt yuv420p -y /home/rikut/py_venvs/my_sim_project/storage/final_sim_dof3_20251217_021445/stimuli/ig_energy/dof3_EL_IGL.mp4


ロード対象ファイル: dof3_EL_IGL
--- ロード完了 ---
Saving animation as '/home/rikut/py_venvs/my_sim_project/storage/final_sim_dof3_20251217_021445/stimuli/ig_energy/dof3_EL_IGL.mp4'...


2025-12-17 12:04:14,104 - matplotlib.animation - INFO - Animation.save using <class 'matplotlib.animation.FFMpegWriter'>
2025-12-17 12:04:14,105 - matplotlib.animation - INFO - MovieWriter._run: running command: ffmpeg -f rawvideo -vcodec rawvideo -s 640x480 -pix_fmt rgba -framerate 30 -loglevel error -i pipe: -vcodec h264 -pix_fmt yuv420p -y /home/rikut/py_venvs/my_sim_project/storage/final_sim_dof3_20251217_021445/stimuli/ig_energy/dof3_EH_IGH.mp4


Saved successfully.
ロード対象ファイル: dof3_EH_IGH
--- ロード完了 ---
Saving animation as '/home/rikut/py_venvs/my_sim_project/storage/final_sim_dof3_20251217_021445/stimuli/ig_energy/dof3_EH_IGH.mp4'...


2025-12-17 12:04:17,791 - matplotlib.animation - INFO - Animation.save using <class 'matplotlib.animation.FFMpegWriter'>
2025-12-17 12:04:17,793 - matplotlib.animation - INFO - MovieWriter._run: running command: ffmpeg -f rawvideo -vcodec rawvideo -s 640x480 -pix_fmt rgba -framerate 30 -loglevel error -i pipe: -vcodec h264 -pix_fmt yuv420p -y /home/rikut/py_venvs/my_sim_project/storage/final_sim_dof3_20251217_021445/stimuli/ig_energy/dof3_EH_IGL.mp4


...

Saved successfully.
ロード対象ファイル: dof3_EL_IGH
--- ロード完了 ---
Saving animation as '/home/rikut/py_venvs/my_sim_project/storage/final_sim_dof3_20251217_021445/stimuli/ig_energy/dof3_EL_IGH.mp4'...


2025-12-17 12:04:35,272 - matplotlib.animation - INFO - Animation.save using <class 'matplotlib.animation.FFMpegWriter'>
2025-12-17 12:04:35,273 - matplotlib.animation - INFO - MovieWriter._run: running command: ffmpeg -f rawvideo -vcodec rawvideo -s 640x480 -pix_fmt rgba -framerate 30 -loglevel error -i pipe: -vcodec h264 -pix_fmt yuv420p -y /home/rikut/py_venvs/my_sim_project/storage/final_sim_dof3_20251217_021445/stimuli/ig_energy/dof3_EL_IGM.mp4


Saved successfully.
ロード対象ファイル: dof3_EL_IGM
--- ロード完了 ---
Saving animation as '/home/rikut/py_venvs/my_sim_project/storage/final_sim_dof3_20251217_021445/stimuli/ig_energy/dof3_EL_IGM.mp4'...


2025-12-17 12:04:38,639 - matplotlib.animation - INFO - Animation.save using <class 'matplotlib.animation.FFMpegWriter'>
2025-12-17 12:04:38,640 - matplotlib.animation - INFO - MovieWriter._run: running command: ffmpeg -f rawvideo -vcodec rawvideo -s 640x480 -pix_fmt rgba -framerate 30 -loglevel error -i pipe: -vcodec h264 -pix_fmt yuv420p -y /home/rikut/py_venvs/my_sim_project/storage/final_sim_dof3_20251217_021445/stimuli/ig_energy/dof3_EH_IGM.mp4


Saved successfully.
ロード対象ファイル: dof3_EH_IGM
--- ロード完了 ---
Saving animation as '/home/rikut/py_venvs/my_sim_project/storage/final_sim_dof3_20251217_021445/stimuli/ig_energy/dof3_EH_IGM.mp4'...
Saved successfully.


150

In [20]:
# フォルダ内のすべてのpklを読み込んで処理する
folder = "/home/rikut/py_venvs/my_sim_project/storage/final_sim_dof3_20251217_021445/stimuli/ig_energy"
folder_path = Path(folder)

loaded_data = []

metrics ={}

for file_path in folder_path.iterdir():
    if file_path.is_file():
            
            # 2. 拡張子がないかチェック
            # .suffix が空文字列（''）であれば拡張子がない
            if file_path.suffix == "":
                
                # 拡張子がないファイルをpklとしてロード
                print(f"ロード対象ファイル: {file_path.name}")
                stim_id = file_path.stem
                
                try:
                    with open(file_path, 'rb') as f:
                        result = pickle.load(f)
                        loaded_data.append(result)
                    print("--- ロード完了 ---")

                    metrics[stim_id] = result.final_metrics
                    




                except pickle.UnpicklingError as e:
                    # pklファイルとして無効な場合のエラー処理
                    print(f"警告: ファイル '{file_path.name}' は有効なPKLファイルではありません。スキップします。エラー: {e}")
                except Exception as e:
                    # その他のファイル読み取りエラー処理
                    print(f"警告: ファイル '{file_path.name}' の読み込み中に予期せぬエラーが発生しました。スキップします。エラー: {e}")

df_met = pd.DataFrame(metrics)
met_path = folder_path / "final_metrics.csv"
df_met.to_csv(met_path)
met_path

ロード対象ファイル: dof3_EL_IGL
--- ロード完了 ---
ロード対象ファイル: dof3_EH_IGH
--- ロード完了 ---
ロード対象ファイル: dof3_EH_IGL
--- ロード完了 ---
ロード対象ファイル: dof3_EM_IGL
--- ロード完了 ---
ロード対象ファイル: dof3_EM_IGM
--- ロード完了 ---
ロード対象ファイル: dof3_EM_IGH
--- ロード完了 ---
ロード対象ファイル: dof3_EL_IGH
--- ロード完了 ---
ロード対象ファイル: dof3_EL_IGM
--- ロード完了 ---
ロード対象ファイル: dof3_EH_IGM
--- ロード完了 ---


PosixPath('/home/rikut/py_venvs/my_sim_project/storage/final_sim_dof3_20251217_021445/stimuli/ig_energy/final_metrics.csv')

In [19]:
df_met.head()

,dof3_EL_IGL,dof3_EH_IGH,dof3_EH_IGL,dof3_EM_IGL,dof3_EM_IGM,dof3_EM_IGH,dof3_EL_IGH,dof3_EL_IGM,dof3_EH_IGM
ig,144.816478,1.706212e+02,1.436982e+02,1.441418e+02,1.567778e+02,1.716546e+02,169.981031,1.549063e+02,1.573734e+02
energy,750.288004,3.753400e+03,3.748234e+03,2.267936e+03,2.267394e+03,2.401382e+03,747.517474,7.985501e+02,3.771199e+03
vfe,391.954329,5.940973e+02,5.612485e+02,5.721301e+02,5.705181e+02,5.776414e+02,535.486548,5.391327e+02,6.206435e+02
jerk,4287.889695,2.278293e+04,3.467922e+04,3.432609e+04,1.638688e+04,2.864874e+04,13466.129187,1.909683e+04,2.518405e+04
torque_change,709173.482782,1.771445e+07,7.070909e+06,3.940730e+06,4.889615e+06,1.647741e+07,536139.978694,3.812872e+06,2.860690e+07
